# 05F — Main experiment on the FULL data: 5 models × 9 datasets × 2 feature sets = 90 runs
Same design as notebook 05, but on all **1,088,589** normal transactions instead of the 50k sample (supervisor's request).

The split is stratified by **error type** (every error type is split exactly 70/30), blanked values are median-imputed **inside each model pipeline** (fitted on the training split only), and each trained model also scores the **1,411 money-laundering transactions** excluded from the experiment: the share it flags is a false-alarm rate on genuine but unusual transactions (`extra_flag_rate`).

**Safe run order** (each phase uses all 9 datasets):
1. **Fast models** – Random Forest, XGBoost, Isolation Forest (54 runs, ~30 min)
2. **LOF** (18 runs, ~3.5 h)
3. **One-Class SVM** (18 runs, ~30 h)

**A fresh run every time:** the notebook first deletes any earlier full-experiment results, so all 90 runs are computed from scratch. Each run is still saved to `results/full_experiment/` the moment it finishes, so partial results are never lost, and the summary cells at the end work with whatever has finished so far.

*Optional:* if a long run is interrupted (e.g. the Mac restarts during One-Class SVM), set `RESUME = True` in the next cell to keep the finished runs and compute only the missing ones. Leave it `False` for the normal, complete run.

Before starting: charger plugged in, lid open, `caffeinate -dims` running in a second Terminal window, other apps closed.

In [ ]:
import sys, time, gc
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # project root, so `config` and `src` can be imported
import config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import prepare as prep
from src.inject import make_dirty
from src.features import reference, build_features, feature_sets
from src.models import run_all

RESUME = False          # False = fresh run (default); True = keep finished runs after an interruption

import shutil
OUT = config.RESULTS / 'full_experiment'
if not RESUME and OUT.exists():
    shutil.rmtree(OUT)                     # start clean: remove results of any earlier run
OUT.mkdir(parents=True, exist_ok=True)
print('Fresh run' if not RESUME else f'Resuming: {len(list(OUT.glob("*.csv")))} runs already finished')
DATASETS = [(r, s) for r in config.RATES for s in config.SEEDS]
PHASES = [['Random Forest', 'XGBoost', 'Isolation Forest'], ['LOF'], ['One-Class SVM']]

def run_file(tag, model, fs):
    return OUT / f"{tag}__{model.replace(' ', '_')}__{fs.replace(' ', '_')}.csv"

## 1. Clean the full dataset once (same steps as notebook 04)

In [ ]:
t = time.time()
raw = prep.load_raw()
normal, laund_raw = prep.remove_laundering(raw), prep.laundering_rows(raw)
del raw; gc.collect()
full, fields = prep.clean_table(normal)                  # field decisions made on normal rows only
laund = prep.clean_like(laund_raw, fields)               # laundering rows cleaned with the SAME decisions
del normal, laund_raw; gc.collect()
ref = reference(full)
print('Full clean table:', full.shape, '| laundering rows:', laund.shape, f'({time.time() - t:.0f}s)')

# Laundering rows -> features (no injected errors; labelled 0 / 'laundering'), built once
XL, _, _ = build_features(laund.assign(is_dq_anomaly=0, issue_type='laundering'), ref)
L_sets = feature_sets(XL)

## 2. Run the 90 runs, phase by phase
For each dataset the dirty data and features are rebuilt in memory (about 30 s), so no large files are written.

In [ ]:
def make_dataset(rate, seed):
    dirty = make_dirty(full, rate=rate, seed=seed)
    X, y, issue = build_features(dirty, ref)
    del dirty; gc.collect()
    return feature_sets(X), y, issue

start = time.time()
print('Started at', time.strftime('%a %H:%M'))
for phase, models in enumerate(PHASES, 1):
    print(f'\n===== Phase {phase}: {", ".join(models)}')
    for rate, seed in DATASETS:
        tag = config.dataset_tag(rate, seed)
        todo = [(m, fs) for m in models for fs in ['with checks', 'raw only'] if not run_file(tag, m, fs).exists()]
        if not todo:
            continue                       # only happens when RESUME = True
        X_sets, y, issue = make_dataset(rate, seed)
        for m, fs in todo:
            t = time.time()
            res, per_issue = run_all({fs: X_sets[fs]}, y, issue, y.mean(), only=[m], extra=L_sets)
            res.insert(0, 'error_rate', rate); res.insert(1, 'seed', seed)
            res['recall_by_issue'] = [per_issue.iloc[0].to_json()]
            res.to_csv(run_file(tag, m, fs), index=False)
            print(f'{tag} | {m:16s} | {fs:11s} | PR-AUC {res["PR-AUC"].iloc[0]:.3f} | '
                  f'{(time.time() - t) / 60:5.1f} min | finished {time.strftime("%a %H:%M")}')
        del X_sets, y, issue; gc.collect()
print(f'\nAll phases done at {time.strftime("%a %H:%M")} ({(time.time() - start) / 3600:.1f} h)')

## 3. Combine the results (works with whatever has finished)

In [ ]:
files = sorted(OUT.glob('*.csv'))
results = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
results.drop(columns='recall_by_issue').to_csv(config.RESULTS / 'full_experiment_all_runs.csv', index=False)
print(f'{len(results)} of 90 runs finished')
print(results.groupby('model').size().rename('runs finished').to_string(), '\n')

summary = (results.groupby(['features', 'model', 'error_rate'])[['PR-AUC', 'precision', 'recall', 'F1', 'seconds']]
           .agg(['mean', 'std']).round(3))
summary.to_csv(config.RESULTS / 'full_experiment_summary.csv')
pr = results.pivot_table(index=['features', 'model'], columns='error_rate', values='PR-AUC', aggfunc=['mean', 'std']).round(3)
pr

## 4. PR-AUC by error rate (mean ± SD over seeds)

In [ ]:
order = ['Random Forest', 'XGBoost', 'One-Class SVM', 'Isolation Forest', 'LOF']
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, fs in zip(axes, ['with checks', 'raw only']):
    sub = results[results.features == fs]
    for m in [m for m in order if m in set(sub.model)]:
        g = sub[sub.model == m].groupby('error_rate')['PR-AUC']
        ax.errorbar(g.mean().index * 100, g.mean(), yerr=g.std(), marker='o', capsize=3, label=m)
    ax.set(title=f'PR-AUC – {fs} (full data)', xlabel='Injected error rate (%)', xticks=[1, 5, 10])
axes[0].set_ylabel('PR-AUC'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(config.RESULTS / 'full_experiment_pr_auc_by_rate.png', dpi=150); plt.show()

## 5. Contribution of the check features (with checks − raw only)

In [ ]:
gain = (results.pivot_table(index=['model', 'error_rate', 'seed'], columns='features', values='PR-AUC')
        .dropna().eval('gain = `with checks` - `raw only`').groupby(['model', 'error_rate'])['gain'].agg(['mean', 'std']).round(3))
gain.unstack('error_rate')

## 6. Recall by error type (averaged over all finished datasets)

In [ ]:
import json
rec = pd.DataFrame([json.loads(r) for r in results['recall_by_issue']])
rec.index = results['model'] + ' | ' + results['features']
rec_mean = rec.groupby(level=0).mean().round(2).rename(columns={'none': 'false-alarm rate (none)'})
rec_mean.to_csv(config.RESULTS / 'full_experiment_recall_by_issue.csv')
rec_mean

## 7. Laundering false-alarm check
Share of the 1,411 excluded money-laundering transactions that each trained model flags as a data-quality error (mean over the finished datasets). These rows contain no injected errors, so every flag is a false alarm on a genuine but unusual transaction.

In [ ]:
laund_rate = (results.groupby(['features', 'model'])['extra_flag_rate'].agg(['mean', 'std']).round(3)
              .rename(columns={'mean': 'laundering flagged (mean)', 'std': 'SD'}))
laund_rate.to_csv(config.RESULTS / 'full_experiment_laundering_false_alarms.csv')
laund_rate

## 8. Full data vs 50k sample (notebook 05)

In [ ]:
sample_file = config.RESULTS / 'experiment_all_runs.csv'
if sample_file.exists():
    key = ['features', 'model', 'error_rate']
    s = pd.read_csv(sample_file).groupby(key)['PR-AUC'].mean().rename('PR-AUC 50k')
    f = results.groupby(key)['PR-AUC'].mean().rename('PR-AUC full')
    comp = pd.concat([s, f], axis=1).dropna()
    comp['difference'] = comp['PR-AUC full'] - comp['PR-AUC 50k']
    display(comp.round(3).unstack('error_rate'))
else:
    print('Run notebook 05 first to compare with the 50k sample.')

## Notes for the paper
- **Main results:** section 3 (mean ± SD of PR-AUC over the three seeds) and the figure in section 4.
- **Run times:** the `seconds` column, the evidence for the computational cost of each model on 1.09M rows.
- **Laundering check:** section 7 (false-alarm rate on genuine but unusual transactions).
- **Sample vs full:** section 8 shows how much each model gains from 20× more data.
- If One-Class SVM was stopped early, report how many of its 18 runs finished and its measured time per run.